# Notebook 9 – Grouping & Aggregation

Grouping and aggregation are used to organize data into meaningful groups and calculate summary statistics for each group. In Pandas, `groupby()` helps us group data based on one or more columns, while aggregation functions such as `sum()`, `mean()`, `count()`, `min()`, and `max()` help us summarize the grouped data. These techniques are widely used in real-world analysis, business reporting, and AI/ML data analysis to identify patterns, compare groups, and generate useful insights.

### The Dataset We'll Use Throughout

We continue with the same online retail customer story from Notebooks 7 and 8 — a single dataset that serves both a business analytics team and an AI/ML team building a **customer churn prediction model**.

- **Real-world / Business angle:** Management doesn't want to look at 100 individual customer rows — they want **summaries by region, membership tier, and churn risk** to decide where to focus marketing spend.
- **AI/ML angle:** Grouped aggregates (e.g., average spend per region, order count per tier) are extremely common **engineered features** in churn models — a customer's value *relative to their group* is often more predictive than their raw numbers alone.

Let's rebuild the (clean) dataset first, then work through each grouping/aggregation technique.

In [6]:
import pandas as pd
import numpy as np
data = {
    "customer_id":   [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "customer_name": ["Aarav", "Priya", "Rahul", "Sneha", "Vikram",
                       "Ananya", "Karthik", "Divya", "Manoj", "Lakshmi"],
    "region":        ["South", "North", "South", "West", "East",
                       "South", "North", "West", "East", "South"],
    "membership":    ["Gold", "Silver", "Gold", "Bronze", "Silver",
                       "Gold", "Bronze", "Gold", "Silver", "Bronze"],
    "total_orders":  [42, 15, 30, 5, 22, 60, 8, 35, 18, 3],
    "total_spend":   [125000, 32000, 98000, 8000, 45000,
                       210000, 12000, 87000, 39000, 4500],
    "last_order_value": [3200, 1500, 4200, 900, 2100,
                          5000, 1100, 3900, 1800, 700],
    "churn_risk":    ["Low", "Medium", "Low", "High", "Medium",
                        "Low", "High", "Low", "Medium", "High"]
}
df = pd.DataFrame(data)
df

,customer_id,customer_name,region,membership,total_orders,total_spend,last_order_value,churn_risk
0,101,Aarav,South,Gold,42,125000,3200,Low
1,102,Priya,North,Silver,15,32000,1500,Medium
2,103,Rahul,South,Gold,30,98000,4200,Low
3,104,Sneha,West,Bronze,5,8000,900,High
4,105,Vikram,East,Silver,22,45000,2100,Medium
5,106,Ananya,South,Gold,60,210000,5000,Low
6,107,Karthik,North,Bronze,8,12000,1100,High
7,108,Divya,West,Gold,35,87000,3900,Low
8,109,Manoj,East,Silver,18,39000,1800,Medium
9,110,Lakshmi,South,Bronze,3,4500,700,High


## 1. `groupby()`

### Concept Explanation
`groupby(column)` splits a DataFrame into groups based on the unique values of one or more columns. On its own it doesn't show output — you attach an aggregation (like `.mean()`, `.sum()`, `.count()`) to compute a summary **per group**. This follows the classic **split → apply → combine** pattern.

### Business + AI/ML Example
The business wants to know the **average total spend per region** to decide which region gets the next marketing budget increase. In ML, this exact "average spend by region" figure is often engineered as a feature (e.g., "customer's spend relative to their region's average") to help a churn model detect under- or over-performing customers.

In [5]:
region_avg_spend = df.groupby("region")["total_spend"].mean()
region_avg_spend

region
East      42000.0
North     22000.0
South    109375.0
West      47500.0
Name: total_spend, dtype: float64

**Output Explanation:** Pandas split the DataFrame into 4 groups (South, North, West, East), computed the mean `total_spend` within each group, and returned one number per region. The **South** region has the highest average spend, largely driven by high spenders like Ananya and Aarav — a clear, actionable business signal.

In [7]:
df.groupby(["region", "membership"])["total_spend"].sum()

region  membership
East    Silver         84000
North   Bronze         12000
        Silver         32000
South   Bronze          4500
        Gold          433000
West    Bronze          8000
        Gold           87000
Name: total_spend, dtype: int64

**Output Explanation:** Grouping by **two columns** (`region`, `membership`) creates a hierarchical (multi-level) result — total spend summed for every unique region+membership combination that actually exists in the data. This is how a business would drill down from "region" to "region and tier" in one step.

## 2. `agg()`

### Concept Explanation
`agg()` (short for aggregate) lets you apply **multiple summary functions at once** — either the same functions across several columns, or different functions per column via a dictionary. It's far more flexible than chaining `.mean()` or `.sum()` alone.

### Business + AI/ML Example
The business wants a single summary table per membership tier showing **order count, total spend, and average spend** together — the kind of table that goes straight into a quarterly report. A data scientist would build the same multi-statistic summary as an aggregated feature table joined back onto each customer row before model training.

In [8]:
tier_summary = df.groupby("membership").agg(
    num_customers=("customer_id", "count"),
    total_orders=("total_orders", "sum"),
    total_spend=("total_spend", "sum"),
    avg_spend=("total_spend", "mean")
).round(2)
tier_summary

,num_customers,total_orders,total_spend,avg_spend
membership,,,,
Bronze,3,16,24500,8166.67
Gold,4,167,520000,130000.00
Silver,3,55,116000,38666.67


**Output Explanation:** `agg()` with named aggregations produced one clean row per membership tier, each with 4 different statistics computed from different source columns in a single call. Notice **Gold** members average far higher spend than Bronze — exactly the kind of tier-comparison table a business reviews when planning loyalty program changes.

## 3. `transform()`

### Concept Explanation
`transform()` also computes a group-level statistic, but — unlike `agg()` — it **broadcasts the result back to the original DataFrame's shape** (one value per row, not one value per group). This means you can attach a group summary directly as a new column next to each individual row.

### Business + AI/ML Example
The business wants to see, right next to each customer, **how their spend compares to their region's average** — without losing any individual customer rows. This "value relative to group average" pattern is one of the single most common engineered features in churn models, since it captures relative behavior, not just absolute numbers.

In [9]:
df["region_avg_spend"] = df.groupby("region")["total_spend"].transform("mean")
df["spend_vs_region_avg"] = (df["total_spend"] - df["region_avg_spend"]).round(2)
df[["customer_name", "region", "total_spend", "region_avg_spend", "spend_vs_region_avg"]]

,customer_name,region,total_spend,region_avg_spend,spend_vs_region_avg
0,Aarav,South,125000,109375.0,15625.0
1,Priya,North,32000,22000.0,10000.0
2,Rahul,South,98000,109375.0,-11375.0
3,Sneha,West,8000,47500.0,-39500.0
4,Vikram,East,45000,42000.0,3000.0
5,Ananya,South,210000,109375.0,100625.0
6,Karthik,North,12000,22000.0,-10000.0
7,Divya,West,87000,47500.0,39500.0
8,Manoj,East,39000,42000.0,-3000.0
9,Lakshmi,South,4500,109375.0,-104875.0


**Output Explanation:** Every row now carries its region's average spend alongside its own spend — notice the same `region_avg_spend` value repeats for all customers in the same region (e.g., all South customers share one value), because `transform()` broadcasts the group result back to every original row. `spend_vs_region_avg` then shows, in plain numbers, whether each customer over- or under-spends relative to their peers — a strong candidate feature for a churn model.

## 4. `filter()` (GroupBy)

### Concept Explanation
GroupBy's `filter(func)` keeps or discards **entire groups** based on a condition applied to each group as a whole (e.g., "keep only groups with more than N members" or "keep only groups whose average exceeds X"). This is different from row-level filtering (Notebook 7) — here the decision is made per-group, and either the whole group stays or the whole group is dropped.

### Business + AI/ML Example
The business wants to focus only on **membership tiers that have more than 2 customers** in this dataset (ignoring any tier that's too small to be statistically meaningful for a report). In ML, this same pattern is used to drop rare categories/groups that don't have enough samples to train a reliable model on.

In [10]:
sufficiently_sized_tiers = df.groupby("membership").filter(lambda g: len(g) > 2)
sufficiently_sized_tiers[["customer_name", "membership"]]

,customer_name,membership
0,Aarav,Gold
1,Priya,Silver
2,Rahul,Gold
3,Sneha,Bronze
4,Vikram,Silver
5,Ananya,Gold
6,Karthik,Bronze
7,Divya,Gold
8,Manoj,Silver
9,Lakshmi,Bronze


**Output Explanation:** `filter()` evaluated `len(g) > 2` (group size greater than 2) for each membership group and kept **every row** belonging to a group that passed. Here, all three tiers qualify (Gold has 4 customers, Silver and Bronze each have 3), so no rows were dropped — but if any tier had only 1 or 2 customers, **all** of its rows would have been removed entirely, unlike row-level filtering (Notebook 7) which evaluates each row independently regardless of group size.

## 5. Pivot Tables

### Concept Explanation
`pivot_table()` reshapes data into a spreadsheet-style summary: choose an `index` (rows), `columns`, `values` to summarize, and an `aggfunc` (default `mean`). It's essentially `groupby()` + reshaping combined into one readable, Excel-like table.

### Business + AI/ML Example
The business wants a classic **Excel-style pivot**: average `total_spend` with **region as rows** and **membership as columns**, so leadership can scan a grid and instantly compare segments. This is also a common way to visually audit a dataset for group imbalances before training a churn model (e.g., spotting a region+tier combination with barely any data).

In [7]:
pivot = pd.pivot_table(
    df,
    index="region",
    columns="membership",
    values="total_spend",
    aggfunc="mean"
)
pivot

membership,Bronze,Gold,Silver
region,,,
East,NaN,NaN,42000.0
North,12000.0,NaN,32000.0
South,4500.0,144333.333333,NaN
West,8000.0,87000.000000,NaN


**Output Explanation:** Each cell shows the average `total_spend` for that specific region+membership combination; `NaN` appears where no customer in the dataset matches that combination (e.g., no West-region Silver customer exists). This grid format is exactly what a business executive expects to see in a spreadsheet-style report, and it also quickly reveals sparse combinations that an ML model would struggle to learn from.

In [11]:
pivot_multi = pd.pivot_table(
    df,
    index="region",
    columns="membership",
    values="total_spend",
    aggfunc=["mean", "count"],
    margins=True,
    margins_name="Overall"
)
pivot_multi

mean                                         count       \
membership        Bronze           Gold        Silver   Overall Bronze Gold   
region                                                                        
East                 NaN            NaN  42000.000000   42000.0    NaN  NaN   
North       12000.000000            NaN  32000.000000   22000.0    1.0  NaN   
South        4500.000000  144333.333333           NaN  109375.0    1.0  3.0   
West         8000.000000   87000.000000           NaN   47500.0    1.0  1.0   
Overall      8166.666667  130000.000000  38666.666667   66050.0    3.0  4.0   

                           
membership Silver Overall  
region                     
East          2.0       2  
North         1.0       2  
South         NaN       4  
West          NaN       2  
Overall       3.0      10

**Output Explanation:** Adding `aggfunc=["mean", "count"]` produces both statistics side by side for every region/membership combination, and `margins=True` adds an "Overall" row and column with grand totals/averages across all groups — giving the business both the detail and the big-picture summary in a single table.

## 6. Crosstab

### Concept Explanation
`pd.crosstab()` builds a **frequency table** — counting how many rows fall into each combination of two (or more) categorical columns. It's similar to a pivot table but defaults to counting occurrences rather than summarizing a numeric column, and is the standard tool for inspecting the relationship between two categorical variables.

### Business + AI/ML Example
The business wants to know **how many customers fall into each region × churn-risk combination**, to see if any region has a concentration of high-risk customers. This exact cross-tabulation is a standard **exploratory data analysis (EDA)** step before training a churn classification model, since it reveals whether churn risk is associated with region (a potential feature relationship worth encoding).

In [9]:
risk_by_region = pd.crosstab(df["region"], df["churn_risk"])
risk_by_region

churn_risk,High,Low,Medium
region,,,
East,0,0,2
North,1,0,1
South,1,3,0
West,1,1,0


**Output Explanation:** Each cell counts how many customers fall into that region+churn-risk combination — for example, South has the most "Low" risk customers, while West and North each contain a "High" risk customer. This kind of frequency table is often the very first check a data scientist runs to see whether a categorical feature (region) looks related to the target (churn risk) before building a model.

In [12]:
risk_by_region_pct = pd.crosstab(df["region"], df["churn_risk"], normalize="index").round(2)
risk_by_region_pct

churn_risk,High,Low,Medium
region,,,
East,0.00,0.00,1.0
North,0.50,0.00,0.5
South,0.25,0.75,0.0
West,0.50,0.50,0.0


**Output Explanation:** Setting `normalize="index"` converts each row into **proportions that sum to 1**, so instead of raw counts we see, for example, what *percentage* of South's customers are Low/Medium/High risk. Percentages like this are easier for business stakeholders to compare across regions of very different sizes, and are a common way to visualize class balance before ML model training.